# Notebook 1: Topic Modelling and Named Entity Recognition

**Module 1 — Pre-NLP: Classical Text Processing**

---

## The Story

You are a data scientist who just joined a media-analytics startup. On day one, your manager drops two CSV files on your desk:

- **2,000 Yelp reviews** — free-form customer feedback, no labels.
- **2,225 BBC news articles** — long-form journalism across five unknown topics.

They ask two questions:

1. **"What are these documents *about*?"** — can we surface the themes without reading every one?
2. **"*Who* and *what* do they mention?"** — which people, companies, and places show up?

No labels, no schema, just raw text. This notebook walks you through the classical toolkit that answers both questions: a text-preprocessing pipeline, Named Entity Recognition (NER) with spaCy, and two flavours of topic modelling — classical **LDA** and modern **BERTopic**.

## Learning Objectives

By the end of this notebook you will be able to:

1. Articulate **why natural language is computationally hard** (ambiguity, polysemy, compositionality).
2. Build a **full text preprocessing pipeline**: lowercase, regex cleanup, tokenize, remove stopwords, lemmatize.
3. Use **spaCy** to tokenize, POS-tag, lemmatize, and run **Named Entity Recognition**.
4. Fit an **LDA topic model** with scikit-learn and interpret topic-word and document-topic matrices.
5. Fit a **BERTopic** model and compare topic quality against LDA.
6. **Choose** between LDA and BERTopic based on corpus size, interpretability, and compute budget.

## Prerequisites

- Python 3.10+, pandas, numpy, basic scikit-learn.
- PyTorch primer is **not strictly required** for this notebook (the deep-learning notebooks later in the course assume it though — see `foundations-genai/PytorchPrimer/`).
- ~30 minutes on Colab's free-tier CPU.

## What you will *not* do here

You will not train a classifier. That is Notebook 2. Today we are in the **unsupervised** world: exploring, describing, and extracting structure from raw text.

## Section 0 — Environment Setup

We need a handful of packages. In Colab, run the next cell to install them. If you are running locally in a virtualenv, you can skip the `!pip install` (but make sure these packages are in your `requirements.txt`).

In [ ]:
# Install required packages (run this first on Google Colab)
# If running locally with a venv, you can skip this cell.

!pip install -q "spacy>=3.7,<3.8" "bertopic>=0.16" "scikit-learn>=1.4" \
    textblob gensim pandas matplotlib wordcloud seaborn

# Download the small English spaCy model. This is the model we will use for
# tokenization, POS-tagging, lemmatization, and NER throughout the notebook.
!python -m spacy download en_core_web_sm

In [ ]:
# ---- Standard library ----
import re
import random
import warnings
from collections import Counter

# ---- Scientific stack ----
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ---- NLP libraries ----
import spacy
from spacy import displacy
from textblob import TextBlob

# ---- Machine learning ----
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# ---- Topic modelling (modern) ----
# BERTopic bundles sentence-transformers + UMAP + HDBSCAN under the hood.
from bertopic import BERTopic

# ---- Word cloud for visual inspection ----
from wordcloud import WordCloud

# ---- Housekeeping ----
warnings.filterwarnings("ignore")

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
# NOTE: LDA in sklearn is not fully deterministic across versions even with
# random_state. That is fine — we care about the *shape* of the topics, not
# exact ordering.

# ---- Global hyperparameters (change these here, not deeper in the notebook) ----
CORPUS_SIZE = 2000                # how many Yelp reviews we sample for speed
N_TOPICS_LDA = 10                 # how many topics LDA will hunt for
TOP_WORDS_PER_TOPIC = 10          # words shown per topic in the summary tables
BERTOPIC_MIN_TOPIC_SIZE = 10      # minimum documents per topic in BERTopic

print(f"spaCy version: {spacy.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")
print(f"random seed: {SEED}")
print("Environment ready.")

## Section 1 — Quick sanity check: spaCy in action

Before anything else, let's confirm the spaCy model loads and does something interesting. We will feed it a single sentence and ask for part-of-speech tags and lemmas.

```python
nlp = spacy.load("en_core_web_sm")
doc = nlp("Apple is looking at buying U.K. startup for $1 billion")
for token in doc:
    print(token.text, token.pos_, token.lemma_)
```

If this cell fails with `OSError: Can't find model 'en_core_web_sm'`, go back and rerun the install cell — the model download is separate from `pip install spacy`.

In [ ]:
# Load the small English pipeline. This object is reusable — load once, use many.
nlp = spacy.load("en_core_web_sm")

# Run the pipeline on one sentence. `doc` is a sequence of Token objects.
sample = "Apple is looking at buying U.K. startup for $1 billion"
doc = nlp(sample)

# Build a small DataFrame so the output is readable.
rows = [(t.text, t.lemma_, t.pos_, t.tag_, t.is_alpha, t.is_stop) for t in doc]
pd.DataFrame(rows, columns=["text", "lemma", "pos", "tag", "is_alpha", "is_stop"])

## Section 1.1 — Why is NLP hard?

Computers eat numbers. Human language is messy, ambiguous, and context-dependent. A few canonical headaches:

1. **Lexical ambiguity** — *"I saw her duck."* Did she duck (verb) or did I see her pet duck (noun)?
2. **Parsing ambiguity** — *"Time flies like an arrow; fruit flies like a banana."* Same surface structure, radically different parse.
3. **Pronoun resolution** — *"The trophy doesn't fit in the suitcase because it is too big."* What is *it*?
4. **Negation & sarcasm** — *"This place was *so* great, I'll never come back."*

Topic modelling and NER do not magically solve these. They handle them *probabilistically* — they get most cases right, fail on edge cases, and that failure mode is why we inspect the output instead of trusting it blindly.

Let's see an ambiguity example in action.

In [ ]:
# Classic headline ambiguity: is "flies" a verb or a noun?
sentences = [
    "Time flies like an arrow",
    "Fruit flies like a banana",
]
for s in sentences:
    doc = nlp(s)
    tags = [(t.text, t.pos_) for t in doc]
    print(s, "->", tags)

# Notice how "flies" gets a different POS tag depending on surrounding words.
# That is spaCy's statistical parser resolving the ambiguity with context.

## Section 2 — Preprocessing Pipeline

Now we load the **Yelp reviews** corpus from a public Dropbox URL. Columns:

- `stars` — 1 to 5 rating
- `text` — free-form review

We subsample to `CORPUS_SIZE` (default: 2,000) for speed. On a full corpus the same code works, just slower.

In [ ]:
# Load Yelp reviews. dl=1 forces Dropbox to serve the raw CSV (not an HTML landing page).
YELP_URL = "https://www.dropbox.com/s/xds4lua69b7okw8/yelp.csv?dl=1"
yelp = pd.read_csv(YELP_URL)

# Subsample with a fixed seed so the notebook is reproducible.
yelp = yelp.sample(n=CORPUS_SIZE, random_state=SEED).reset_index(drop=True)

print(f"Yelp shape: {yelp.shape}")
print(f"Columns: {list(yelp.columns)}")
yelp.head(3)

### 2.1 Regex cleanup

Before tokenization we strip things that are not language: URLs, non-ASCII characters, excess whitespace, and most punctuation.

A minimal cleanup looks like this:

```python
import re
text = re.sub(r"http\S+", " ", text)      # kill URLs
text = re.sub(r"[^a-zA-Z\s]", " ", text) # keep only letters and whitespace
text = re.sub(r"\s+", " ", text).strip()  # collapse whitespace
```

We will wrap this up into a `clean_text` helper and reuse it.

In [ ]:
def clean_text(text: str) -> str:
    """Aggressive but predictable text cleanup.

    Steps:
      1. Lowercase.
      2. Remove URLs (http, https, bit.ly, etc.).
      3. Drop non-letter characters (keeps words, discards numbers/punctuation).
      4. Collapse repeated whitespace.
    """
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Sanity-check on a real review.
raw = yelp["text"].iloc[0]
print("RAW:\n", raw[:300], "\n")
print("CLEAN:\n", clean_text(raw)[:300])

### 2.2 Tokenization: spaCy vs TextBlob

Two popular tokenizers:

- **spaCy** — respects contractions, punctuation, MWEs via its statistical pipeline.
- **TextBlob** — simpler, regex-driven, `TextBlob(text).words`.

For topic modelling on English, spaCy is the safer default. TextBlob is fine for quick exploratory work.

In [ ]:
sample = "It's the best coffee shop I've ever been to — 10/10 would visit again!"

# spaCy tokenization
spacy_tokens = [t.text for t in nlp(sample)]
# TextBlob tokenization
textblob_tokens = list(TextBlob(sample).words)

print("spaCy    :", spacy_tokens)
print("TextBlob :", textblob_tokens)

# Note how spaCy keeps punctuation as its own tokens while TextBlob drops them.
# Neither is "right" — it depends on the downstream task.

### 2.3 Stopword removal

Stopwords are very-high-frequency words that carry little topical meaning: *the, a, is, and, of…*. Removing them speeds up downstream models and sharpens topics.

spaCy ships with a default English stopword list at `spacy.lang.en.stop_words.STOP_WORDS`. Often you want to **extend** it with domain-specific stopwords — for Yelp reviews, words like *restaurant, place, food* appear in almost every document and will dominate LDA topics if you leave them in.

```python
from spacy.lang.en.stop_words import STOP_WORDS
custom_stop = STOP_WORDS | {"restaurant", "place", "food"}
```

In [ ]:
from spacy.lang.en.stop_words import STOP_WORDS

# Extend spaCy's default list with Yelp-specific filler words that would otherwise
# dominate every topic.
CUSTOM_STOPWORDS = STOP_WORDS | {
    "restaurant", "place", "food", "like", "good", "great",
    "time", "just", "really", "nice", "got", "came", "went",
}

sample_tokens = ["the", "pizza", "at", "this", "restaurant", "was", "delicious"]
filtered = [t for t in sample_tokens if t not in CUSTOM_STOPWORDS]
print("before:", sample_tokens)
print("after :", filtered)

print(f"\nDefault stopword list size: {len(STOP_WORDS)}")
print(f"Custom stopword list size:  {len(CUSTOM_STOPWORDS)}")

### 2.4 Lemmatization vs stemming

Both map word variants to a canonical form.

- **Stemming** (rule-based): *running → run*, *better → better*. Fast, crude, can produce non-words (*universities → univers*).
- **Lemmatization** (dictionary-based): *running → run*, *better → good*, *mice → mouse*. Slower but correct.

For topic modelling, lemmatization matters a lot: if *"good"* and *"better"* are treated as different tokens, they split their probability mass across two topics and neither is clearly *"positive sentiment"*.

In [ ]:
# spaCy lemmatization
doc = nlp("The mice were running faster than the better cats.")
for t in doc:
    print(f"{t.text:10s} -> {t.lemma_}")

# Observe: `mice` -> `mouse`, `were` -> `be`, `running` -> `run`, `better` -> `well`
# This normalization is exactly what we want before counting words for LDA.

### Lab 2.1 — Build the full `preprocess` function

Your task: combine everything above into a single `preprocess(text)` function that:

1. Lowercases and regex-cleans the text (use `clean_text`).
2. Parses the cleaned text with `nlp`.
3. Keeps only **alphabetic** tokens (`token.is_alpha`) that are not in `CUSTOM_STOPWORDS` (compare against `token.lemma_`).
4. Returns a **space-joined string of lemmas**.

Then apply it to the full Yelp corpus and store the result in `yelp["clean"]`.

**Tips**
- Use `token.lemma_`, not `token.text`, so that *"running"* and *"runs"* both collapse to *"run"*.
- Reject very short lemmas (len < 3) — they rarely help topic modelling.
- Expect ~30 seconds runtime on 2,000 reviews.

In [ ]:
def preprocess(text: str) -> str:
    """Full preprocessing pipeline: clean + tokenize + stopword-remove + lemmatize."""

    # 1. Basic regex cleanup (reuse the helper we already built)
    text = None  # YOUR CODE

    # 2. Parse with spaCy
    doc = None  # YOUR CODE

    # 3. Keep alphabetic, non-stopword lemmas longer than 2 chars
    #    Hint: iterate over `doc`, use `t.is_alpha`, `t.lemma_`, filter on CUSTOM_STOPWORDS.
    tokens = None  # YOUR CODE

    # 4. Join back into a single string (what CountVectorizer expects)
    return None  # YOUR CODE


# Test on a single review first — it should print a clean, space-separated string.
example = yelp["text"].iloc[0]
print("RAW   :", example[:200])
# print("CLEAN :", preprocess(example)[:200])  # uncomment when your function works

Now apply `preprocess` to the whole Yelp corpus. On 2,000 reviews this takes ~30 seconds.

For serious workloads you would use `nlp.pipe(texts, batch_size=64)` which is 2-5x faster, but for 2k docs the simple `.apply` is fine and easier to read.

In [ ]:
# Apply the pipeline. This will take ~30s.
yelp["clean"] = yelp["text"].apply(preprocess)

# Sanity check: first 3 cleaned reviews.
for i in range(3):
    print(f"[{i}] {yelp['clean'].iloc[i][:180]}\n")

print(f"Average clean-review length (words): "
      f"{yelp['clean'].str.split().str.len().mean():.1f}")

## Section 3 — Named Entity Recognition (NER)

**NER** is the task of spotting spans of text that refer to real-world *things* and labelling them — people, companies, places, dates, money.

Why care? Three reasons:

1. **Information extraction** — populate a knowledge graph or search index.
2. **Anonymization** — redact PII (PERSON, GPE) before sharing data.
3. **Faceted exploration** — "which companies are most mentioned in this news corpus?" That's our BBC question from the opening story.

spaCy's `en_core_web_sm` ships with an NER model out of the box. Calling `nlp(text)` already runs it — entities land at `doc.ents`.

Common spaCy entity types:

| Label | Meaning |
|-------|---------|
| `PERSON` | People, including fictional |
| `ORG` | Companies, agencies, institutions |
| `GPE` | Countries, cities, states |
| `LOC` | Non-GPE locations (mountain ranges, bodies of water) |
| `MONEY` | Monetary values |
| `DATE` | Dates and periods |
| `PRODUCT` | Objects, vehicles, foods |

Let's see it on a few Yelp reviews.

In [ ]:
# Run NER on the first 5 raw reviews (we want the ORIGINAL casing and punctuation
# so the NER model can do its job properly — don't feed in `clean` text here).
ent_rows = []
for idx, review in enumerate(yelp["text"].head(5)):
    doc = nlp(review)
    for ent in doc.ents:
        ent_rows.append({
            "review_id": idx,
            "entity": ent.text,
            "label": ent.label_,
        })

ent_df = pd.DataFrame(ent_rows)
ent_df

### 3.1 Visual inspection with displaCy

spaCy ships with an inline visualizer. Running `displacy.render(doc, style='ent')` on a single document renders a coloured HTML block that highlights each entity.

**Warning:** never run displaCy on a whole corpus — it renders one HTML block per document and will freeze your notebook. Cap it to 1–3 docs.

In [ ]:
# Pick the first review, render in-line. Safe because it's exactly ONE document.
doc = nlp(yelp["text"].iloc[0])
displacy.render(doc, style="ent", jupyter=True)

### 3.2 NER on the BBC corpus

Time for the second half of the opening story. We load the **BBC news** corpus (2,225 articles, 5 known categories) and ask: *which organizations show up most?*

> The BBC dataset is labelled (sport/business/tech/entertainment/politics). We deliberately **ignore** the labels for now — we are in the unsupervised regime. Later (in the lab) we will quietly peek at the labels to sanity-check whether LDA recovered anything real.

In [ ]:
BBC_URL = "https://www.dropbox.com/scl/fi/lfa2ryv86uqd3y988irfw/bbc.csv?rlkey=vtwdf6g8sejhkf75p7o36ev00&dl=1"
bbc = pd.read_csv(BBC_URL)

print(f"BBC shape: {bbc.shape}")
print(f"Columns  : {list(bbc.columns)}")
print("\nCategory counts (for later — we do NOT use these until we evaluate LDA):")
print(bbc["category"].value_counts())
bbc.head(2)

We use `nlp.pipe` here because we are about to run the pipeline on 2,225 full articles. `pipe` batches internally and is dramatically faster than calling `nlp(text)` 2,225 times in a loop.

For time, we take the first 500 BBC articles. Feel free to bump this up if you have patience.

In [ ]:
N_BBC_FOR_NER = 500
org_counter = Counter()

# nlp.pipe returns a generator of Doc objects. disable the parser for a speedup —
# we only need the NER component here.
for doc in nlp.pipe(bbc["text"].head(N_BBC_FOR_NER), batch_size=32,
                    disable=["parser", "tagger", "lemmatizer"]):
    for ent in doc.ents:
        if ent.label_ == "ORG":
            org_counter[ent.text.strip()] += 1

top_orgs = org_counter.most_common(20)
print("Top 20 organizations mentioned in the first 500 BBC articles:")
for name, count in top_orgs:
    print(f"  {count:4d}  {name}")

In [ ]:
# Bar chart of the top 15 orgs. This is the artefact you'd show a stakeholder:
# "here are the top entities in your news feed."
top15 = dict(org_counter.most_common(15))

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(list(top15.keys())[::-1], list(top15.values())[::-1], color="steelblue")
ax.set_xlabel("Mentions")
ax.set_title("Top 15 organizations in BBC news (first 500 articles)")
plt.tight_layout()
plt.show()

### Lab 3.1 — Extract top PERSON entities

Your turn. Using the same `nlp.pipe` pattern:

1. Iterate over the first 500 BBC articles.
2. Collect every entity whose label is `PERSON`.
3. Count occurrences in a `Counter`.
4. Print the **top 10**.

Hints:
- `ent.label_ == "PERSON"` is the filter.
- Use `.strip()` on the text to avoid "Tony Blair " and "Tony Blair" counting separately.
- Expect politicians and celebrities to dominate.

In [ ]:
person_counter = Counter()

# 1. Iterate over the BBC corpus using nlp.pipe for speed.
#    Hint: disable parser/tagger/lemmatizer — we only want NER here.
for doc in None:  # YOUR CODE  -- replace None with the nlp.pipe(...) call
    # 2. For every entity in the doc, keep only PERSON entities.
    for ent in doc.ents:
        if None:  # YOUR CODE  -- check the label
            person_counter[None] += 1  # YOUR CODE  -- what do we store?

# 3. Print top 10
top_people = None  # YOUR CODE
print(top_people)

## Section 4 — Topic Modelling with LDA

**Latent Dirichlet Allocation** is the classical topic model. The generative story:

1. Each document is a **mixture of topics** (e.g., 60% sports, 30% business, 10% politics).
2. Each topic is a **distribution over words** (the "sports" topic puts high probability on *match, goal, team*).
3. To generate a word in a document, you first pick a topic (from the document's mixture), then pick a word (from the topic's distribution).

LDA inverts this process: given the corpus, infer the topic distributions and document mixtures that most likely generated it.

**Intuition via a toy corpus** (2 topics — sports and politics):

```
doc 1: "the team won the match with a late goal"          -> 90% sports, 10% politics
doc 2: "the president announced a new policy today"       -> 10% sports, 90% politics
doc 3: "the team's coach endorsed the president's policy" -> 50% sports, 50% politics
```

LDA is bag-of-words — it does not care about word order. That's its main weakness and BERTopic's main selling point.

### What we will do

1. Turn the cleaned BBC articles into a **document-term matrix** with `CountVectorizer`.
2. Fit `LatentDirichletAllocation(n_components=N_TOPICS_LDA)`.
3. Inspect `lda.components_` → which words dominate each topic?
4. Inspect `lda.transform(X)` → which topic does each document belong to?
5. **Lab:** does LDA recover the 5 ground-truth BBC categories?

In [ ]:
# First, preprocess the BBC corpus with the same pipeline we used on Yelp.
# This takes ~3-4 minutes on CPU — it's the slowest cell in the notebook.
# If short on time, subsample first.

print("Preprocessing BBC corpus (this takes a few minutes)...")
bbc["clean"] = bbc["text"].apply(preprocess)
print("Done. Sample cleaned article:")
print(bbc["clean"].iloc[0][:300])

In [ ]:
# Build the document-term matrix.
# min_df=5: a word must appear in at least 5 docs to be counted (kills rare junk)
# max_df=0.5: drop words appearing in more than 50% of docs (kills corpus-wide filler)
vectorizer = CountVectorizer(min_df=5, max_df=0.5)
X_bbc = vectorizer.fit_transform(bbc["clean"])
print(f"Document-term matrix shape: {X_bbc.shape}  (docs x vocabulary)")

# Fit LDA. n_components=5 because we secretly know BBC has 5 categories.
# In a real unsupervised setting you would try k ∈ {5, 10, 20} and inspect.
lda = LatentDirichletAllocation(
    n_components=5,
    random_state=SEED,
    max_iter=20,
    learning_method="batch",
)
lda.fit(X_bbc)
print("LDA fitted.")

In [ ]:
def top_words_per_topic(model, feature_names, n_top=10):
    """Return a DataFrame: one column per topic, rows are top words."""
    rows = {}
    for topic_idx, weights in enumerate(model.components_):
        top_indices = weights.argsort()[:-n_top - 1:-1]
        rows[f"Topic {topic_idx}"] = [feature_names[i] for i in top_indices]
    return pd.DataFrame(rows)

feature_names = vectorizer.get_feature_names_out()
top_words_per_topic(lda, feature_names, n_top=TOP_WORDS_PER_TOPIC)

### 4.1 Document-topic distribution

`lda.transform(X)` gives a (n_docs, n_topics) matrix where each row sums to 1 — the mixture weights for that document.

Below we show the mixture for 3 random articles. A "clean" topic separation would have one dominant weight (~0.9) per row.

In [ ]:
doc_topic = lda.transform(X_bbc)
print(f"Document-topic matrix shape: {doc_topic.shape}\n")

for i in [0, 500, 1500]:
    weights = doc_topic[i]
    dominant = weights.argmax()
    print(f"Article {i}  (true category: {bbc['category'].iloc[i]})")
    print(f"  Topic weights: {np.round(weights, 2)}")
    print(f"  Dominant topic: {dominant}")
    print()

### 4.2 Visualize topics as word clouds

A word cloud per topic is a quick "story in a picture". Word size is proportional to topic-word weight.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for topic_idx, ax in enumerate(axes):
    # Build {word: weight} for this topic
    weights = lda.components_[topic_idx]
    top_indices = weights.argsort()[:-30 - 1:-1]
    word_weights = {feature_names[i]: weights[i] for i in top_indices}

    wc = WordCloud(width=400, height=300, background_color="white").generate_from_frequencies(word_weights)
    ax.imshow(wc, interpolation="bilinear")
    ax.set_title(f"Topic {topic_idx}")
    ax.axis("off")

plt.tight_layout()
plt.show()

### Lab 4.1 — Does LDA recover the BBC categories?

**This is the main homework of the notebook.**

BBC has 5 labelled categories (sport, business, tech, entertainment, politics). We fitted LDA with 5 topics. The question: *did LDA rediscover the human labels?*

Your steps:

1. Assign each document to its **most likely topic** (use `np.argmax` on `doc_topic`).
2. Build a **cross-tabulation** of (predicted topic) vs (true category). pandas' `pd.crosstab` is perfect.
3. Visualize it as a heatmap. A "clean recovery" shows one big cell per row.
4. **Write a 1-sentence answer** in a markdown cell: did LDA recover the categories? Which topic maps to which category?

Expected outcome: usually 3-4 of the 5 categories are cleanly recovered; the remaining 1-2 get merged (business and politics often bleed into each other).

In [ ]:
# 1. Dominant topic per document
#    Hint: np.argmax over axis=1 of doc_topic
dominant_topic = None  # YOUR CODE

# 2. Cross-tabulate: rows = dominant topic, columns = true category
#    Hint: pd.crosstab(dominant_topic, bbc['category'])
crosstab = None  # YOUR CODE

print(crosstab)

# 3. Heatmap
fig, ax = plt.subplots(figsize=(8, 5))
# YOUR CODE: call sns.heatmap on `crosstab` with annot=True, fmt='d', cmap='Blues'
None
ax.set_title("LDA topic vs BBC true category (higher = better recovery)")
plt.tight_layout()
plt.show()

## Section 5 — Topic Modelling with BERTopic

LDA's fundamental limitation: it treats text as a bag of independent words. *"not good"* has the same bag-of-words as *"good not"*. Contextual meaning is gone.

**BERTopic** fixes this by:

1. Embedding each document with a **sentence-transformer** (e.g., MiniLM) — each doc becomes a dense vector that *does* respect word order and context.
2. Reducing dimensionality with **UMAP** (768-dim → ~5-dim).
3. Clustering with **HDBSCAN**, which naturally finds an unknown number of clusters and rejects noise (cluster `-1`).
4. Extracting representative words per cluster with a custom TF-IDF.

Trade-offs vs LDA:

| aspect | LDA | BERTopic |
|---|---|---|
| speed | fast | slow (embeddings + UMAP) |
| quality on long docs | good | great |
| quality on short docs (tweets) | poor | usually better |
| needs GPU? | no | optional (CPU works) |
| interpretability | easy (top words) | easy (top words per cluster) |
| number of topics | you pick | HDBSCAN picks |
| small corpus (<100 docs) | works | tends to assign everything to -1 |

In [ ]:
# Fit BERTopic on the cleaned BBC corpus.
# For speed we use the default MiniLM embedder. On Colab free tier this takes
# about 2-3 minutes for 2,225 docs.
# NOTE: pass the RAW (not `clean`) text here — sentence-transformers benefit
# from natural language including stopwords and punctuation.

topic_model = BERTopic(
    min_topic_size=BERTOPIC_MIN_TOPIC_SIZE,
    nr_topics=6,            # 5 real categories + 1 outlier bucket
    verbose=False,
)
bert_topics, bert_probs = topic_model.fit_transform(bbc["text"])
print("BERTopic fitted. Discovered topics (including outlier -1):")
print(topic_model.get_topic_info().head(10))

### 5.1 Inspect BERTopic topics

`topic_model.get_topic(i)` returns the representative `(word, score)` pairs for topic `i`. Topic `-1` is the outlier bucket — documents that HDBSCAN couldn't confidently assign to any cluster.

In [ ]:
# Show the representative words for each non-outlier topic.
info = topic_model.get_topic_info()

for topic_id in info["Topic"].head(8):
    if topic_id == -1:
        continue
    top = topic_model.get_topic(topic_id)
    words = ", ".join(w for w, _ in top[:TOP_WORDS_PER_TOPIC])
    print(f"Topic {topic_id:2d}: {words}")

### 5.2 Compare LDA vs BERTopic on the same corpus

Side-by-side, do the topic-word lists feel more or less coherent?

**What to look for:**

- **Coherence:** do the top words in one topic actually co-occur in plausible real-world contexts?
- **Redundancy:** do two topics look essentially the same? LDA is more prone to this than BERTopic.
- **Specificity:** BERTopic tends to surface finer-grained topics (e.g., *"mobile phones"* vs *"tech"*).

In [ ]:
print("=" * 60)
print("LDA (5 topics)")
print("=" * 60)
for topic_idx in range(5):
    top_indices = lda.components_[topic_idx].argsort()[:-6 - 1:-1]
    words = ", ".join(feature_names[i] for i in top_indices)
    print(f"Topic {topic_idx}: {words}")

print()
print("=" * 60)
print("BERTopic (first 5 non-outlier topics)")
print("=" * 60)
shown = 0
for topic_id in topic_model.get_topic_info()["Topic"]:
    if topic_id == -1:
        continue
    words = ", ".join(w for w, _ in topic_model.get_topic(topic_id)[:6])
    print(f"Topic {topic_id}: {words}")
    shown += 1
    if shown >= 5:
        break

### Lab 5.1 — End-to-end self-check

Given one brand-new review, build a tiny function that returns:

1. The **cleaned text** (via `preprocess`).
2. The **entities** spaCy finds (list of `(text, label)` tuples).
3. The **dominant LDA topic id**.
4. The **dominant BERTopic topic id**.

Use the helpers you already built and the fitted models (`lda`, `vectorizer`, `topic_model`).

In [ ]:
NEW_REVIEW = (
    "Apple announced record earnings today, with CEO Tim Cook saying "
    "iPhone sales in China drove the quarter. Shares rose 4% in New York."
)

def analyze(text: str):
    # 1. Cleaned text
    cleaned = None  # YOUR CODE

    # 2. Entities (list of (text, label) tuples). Use the original `text`, not `cleaned`.
    entities = None  # YOUR CODE

    # 3. LDA dominant topic
    #    Hint: vectorizer.transform([cleaned]) -> lda.transform(...) -> argmax
    lda_topic = None  # YOUR CODE

    # 4. BERTopic dominant topic
    #    Hint: topic_model.transform([text])[0][0]
    bert_topic = None  # YOUR CODE

    return {
        "cleaned": cleaned,
        "entities": entities,
        "lda_topic": lda_topic,
        "bert_topic": bert_topic,
    }

# result = analyze(NEW_REVIEW)
# for k, v in result.items():
#     print(f"{k}: {v}")

## Section 6 — Wrap-up

### What you built

- A **text-preprocessing pipeline** (clean → tokenize → lemmatize → stopword-filter) that you can reuse in every downstream notebook.
- A **NER extractor** that surfaces people, organizations and places from raw text.
- An **LDA topic model** that compresses a corpus into `k` interpretable word distributions.
- A **BERTopic model** that does the same but on top of modern sentence embeddings.

### When to use what

| Situation | Reach for |
|-----------|-----------|
| <500 short docs, need speed, tight compute | LDA |
| News articles / long documents | LDA *or* BERTopic (BERTopic usually wins) |
| Tweets, product reviews (short, noisy) | BERTopic |
| Need human-readable topics for a dashboard | Either — both give top-word lists |
| Need to find "novel" topics without picking k | BERTopic (HDBSCAN picks for you) |

### Self-check quiz

1. **Why does lemmatization matter more than stemming for topic modelling?**
2. If LDA gives you a topic with top words `[game, team, player, match, goal]`, can you confidently name the topic? What about a topic with top words `[good, place, time, great, people]`?
3. Name one case where BERTopic clearly beats LDA. Name one case where LDA is the pragmatic choice.

### What's next

Notebook 2 flips the script: the text still comes in raw, but now we have **labels** (positive / negative Yelp reviews). We'll build our first supervised classifier — Multinomial Naive Bayes — and establish the baseline that every neural model later in the course will have to beat.

### References

- spaCy 101: https://spacy.io/usage/spacy-101
- BERTopic docs: https://maartengr.github.io/BERTopic/
- Blei, Ng & Jordan (2003) — the original LDA paper: https://www.jmlr.org/papers/volume3/blei03a/blei03a.pdf